# Lab 6: Text Retrieval

**Business question:** Does semantic search return evidence that answers the analyst's question?

This lab has three parts:

| Part | Data | Time | What you hand in |
|---|---|---|---|
| 1 Guided core | Four short queries over the 24 customer messages from Lab 1 | about 35 min | Run outputs, saved files and the checkpoint |
| 2 Practice | Five complaint queries over the CFPB training complaints | about 60–90 min | Three exercises (six judgments, a metric table, one failure) and the interpretation |
| 3 Challenge | Both | optional | Nothing required |

Graded with the course Lab Notebook rubric in Canvas.

You do not run a model in this lab. I embedded the queries with the same model as the Module 5 vectors and saved the results. A second model, a **cross-encoder reranker**, also ran once for the optional Challenge, and its scores are saved too. You load these saved files, so everyone gets the same numbers.

## Part 1 Guided core

### 1.1 Load the libraries

Select the `ban6403` kernel, then run the cell below. It loads the libraries we use in this lab.

In [ ]:
from pathlib import Path   # file and folder paths
import json                # reads the manifest files
import time                # times the search in Challenge A4
import numpy as np         # arrays of numbers: the vectors
import pandas as pd        # tables
from sklearn.feature_extraction.text import TfidfVectorizer  # the Module 4 word features

### 1.2 Set the data and output folders

The notebook runs from the `notebooks` folder, so the data is one folder up, in `../data`. `EMB` holds the saved vectors. `RET` holds the retrieval files: the queries, the relevance judgments and the reranker scores. Your own results go to `../outputs/m06`.

In [ ]:
# Folder paths, relative to the notebooks folder.
DATA = Path("../data")
EMB = DATA / "embeddings"
RET = DATA / "retrieval"
OUTPUT_DIR = Path("../outputs/m06")
# Create the output folder if it does not exist yet.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

### 1.3 The query set

A retrieval test starts with a fixed list of **queries**: the questions an analyst would type. Each query has an ID and an **information need**, a sentence that says what a useful result must contain. We use the same queries for every method, so the methods can be compared fairly.

The first four queries (TQ01 to TQ04) search the customer messages.

In [ ]:
queries = pd.read_csv(RET / "m06_queries.csv")

# Look up a query's text or information need by its ID, for example query_text["TQ01"].
query_text = queries.set_index("query_id")["query"]
need = queries.set_index("query_id")["information_need"]

# query_row["TQ01"] gives the row position of a query, which is also its row in the query vectors.
query_row = pd.Series(range(len(queries)), index=queries["query_id"])

# Show the four message queries (query_set "teaching" in the file).
queries.loc[queries["query_set"].eq("teaching"), ["query_id", "query", "information_need"]]

### 1.4 Load the messages, the message vectors and the query vectors

The message vectors are the ones you used in Module 5, one per message with text, in file order. The query vectors were made with the **same model and revision**. A cosine between vectors from two different models means nothing. So we read both manifests and print the model that made each set.

In [ ]:
# The 23 messages with text, numbered 0 to 22 so that row i matches msg_vectors[i].
feedback = pd.read_csv(DATA / "raw/customer_feedback_m01.csv")
messages = feedback.dropna(subset=["text"]).reset_index(drop=True)

# Saved vectors: one row per message and one row per query.
# The files store compact 16-bit numbers; .astype("float32") turns them into ordinary numbers for computing.
msg_vectors = np.load(EMB / "feedback_minilm.npy").astype("float32")
query_vectors = np.load(EMB / "m06_queries_minilm.npy").astype("float32")
print("message vectors:", msg_vectors.shape, "query vectors:", query_vectors.shape)

# Both manifests should name the same model and revision ([:7] shows the first 7 characters).
fb_manifest = json.loads((EMB / "feedback_minilm_manifest.json").read_text())
q_manifest = json.loads((EMB / "m06_queries_minilm_manifest.json").read_text())
print("messages made with:", fb_manifest["model"], fb_manifest["model_revision"][:7])
print("queries made with: ", q_manifest["model"], q_manifest["model_revision"][:7])

### 1.5 Lexical retrieval

**Lexical retrieval** scores each message by the words it shares with the query. We fit TF-IDF on the 23 messages (words and two-word phrases, as in Module 4) and use `transform` for the query. The helper `rank_messages` returns the top `k` messages. A message with a score of 0 or less is not returned; for lexical search, a score of 0 means no shared word. The same rule applies to dense search in 1.6.

Here, all 23 messages form the collection we intend to search, so we fit the index on that collection. For predictive evaluation, fit preprocessing on training data only, as in Module 4.

Query TQ01 asks for "money back for a broken product". Its information need: the customer asks for, or reports, a refund or replacement because a product failed. Keep the need in mind as you read which messages come first.

In [ ]:
# Same TF-IDF setup as Module 4: words and two-word phrases (ngram_range=(1, 2)).
msg_tfidf = TfidfVectorizer(ngram_range=(1, 2))
X_msg = msg_tfidf.fit_transform(messages["text"])

def rank_messages(query_id, method, k=5):
    """Take a query ID and "lexical" or "dense"; return the top k messages with rank and score."""
    if method == "lexical":
        # transform turns the query into a TF-IDF row with the vocabulary of the messages.
        query_tfidf = msg_tfidf.transform([query_text[query_id]])
        # One cosine per message; .ravel() turns the one-column result into a flat list.
        scores = (X_msg @ query_tfidf.T).toarray().ravel()
    else:
        # One dot product (cosine) per message, as in Module 5.
        scores = msg_vectors @ query_vectors[query_row[query_id]]
    found = messages[["feedback_id", "text"]].copy()
    found["score"] = scores
    # Keep scores above 0, sort from the highest down and keep the first k.
    # kind="stable" keeps tied scores in file order, so everyone sees the same list.
    found = found[found["score"] > 0]
    found = found.sort_values("score", ascending=False, kind="stable").head(k)
    # Number the results 1, 2, 3, ... in a new first column.
    found.insert(0, "rank", range(1, len(found) + 1))
    found["score"] = found["score"].round(2)
    return found.reset_index(drop=True)

print("Query:", query_text["TQ01"])
print("Need:", need["TQ01"])
rank_messages("TQ01", "lexical")

### 1.6 Dense retrieval

**Dense retrieval** scores each message by the cosine between its vector and the query vector. All vectors have length 1, so the cosine is the dot product, as in Module 5. Compare this list with 1.5.

In [ ]:
# The same helper as in 1.5, now with the saved vectors instead of TF-IDF.
rank_messages("TQ01", "dense")

### 1.7 Relevance judgments

A ranked list is not evidence until someone decides which results are useful. A **relevance judgment** records that decision for one query and one result: 1 = relevant, 0 = not relevant. The rule we follow:

> A result is **relevant** when its main point matches the information need. Sharing a topic or a word is not enough. Judge the content against the need: a query's product filter is a search setting, not part of the need. A solved problem does not count only when the need asks for an open problem ("not yet fixed", "has not reached them yet"). A need that asks whether the customer reports something is met by a report of a past, solved case. When the text does not say enough to decide, judge it not relevant.

Every message was judged for each of the four message queries. These are draft judgments. Read the passage and compare your reasoning with the `note` column, which gives the reason in a few words. Question any judgment that does not match the rule. Here are the relevant messages for TQ01.

In [ ]:
judgments = pd.read_csv(RET / "m06_relevance_judgments.csv")
# "Int64" is a pandas whole-number type that allows empty values (pairs not judged yet).
judgments["relevant"] = judgments["relevant"].astype("Int64")
print("Information need:", need["TQ01"])

# Rows for TQ01 that are judged relevant (& means both conditions must be true).
judgments[judgments["query_id"].eq("TQ01") & judgments["relevant"].eq(1)]

### 1.8 Precision at 5 and reciprocal rank

We add the judgments to a ranked list, then compute two numbers:

- **Precision at 5 (P@5)**: the share of the top 5 places that hold a relevant result. Empty places count as not relevant.
- **Reciprocal rank at 10 (RR@10)**: 1 divided by the rank of the first relevant result in the top 10, or 0 if there is none. RR@10 = 1 means the first result is relevant; RR@10 = 0.5 means the first relevant result is second. The tables call this column `RR`.

A result without a judgment counts as not relevant here. We check for those later.

In [ ]:
def with_judgments(found, query_id, id_col):
    """Take a ranked list and a query ID; return the list with a "relevant" column (1, 0 or empty)."""
    # The judgments for this query, looked up by document ID.
    query_labels = judgments[judgments["query_id"].eq(query_id)].set_index("doc_id")["relevant"]
    found = found.copy()
    # .map looks up each ID in query_labels; the IDs are stored as text in the judgments file.
    found["relevant"] = found[id_col].astype(str).map(query_labels).astype("Int64")
    return found

def precision_at_k(relevant, k=5):
    """Take the relevant column of a ranked list; return the share of relevant results in the top k."""
    # Empty (not judged) counts as 0. Dividing by k counts empty places as not relevant.
    return relevant.head(k).fillna(0).sum() / k

def reciprocal_rank(relevant):
    """Take the relevant column of a ranked list; return 1 / rank of the first relevant result, or 0."""
    # enumerate(..., start=1) gives each value together with its rank 1, 2, 3, ...
    for rank, value in enumerate(relevant.fillna(0), start=1):
        if value == 1:
            return 1 / rank
    return 0.0

# Rank the top 10 with each method, add the judgments and compute the two metrics.
for method in ["lexical", "dense"]:
    found = with_judgments(rank_messages("TQ01", method, k=10), "TQ01", "feedback_id")
    print(method, "| P@5:", precision_at_k(found["relevant"]), "| RR:", round(reciprocal_rank(found["relevant"]), 2))

### 1.9 A metric table for all four queries

Now the same two numbers for all four message queries and both methods. The column `relevant_total` counts the relevant messages for the query. With only two relevant messages, P@5 cannot be higher than 0.4. Read P@5 together with that count.

In [ ]:
# One row per query and method.
rows = []
for query_id in ["TQ01", "TQ02", "TQ03", "TQ04"]:
    # Number of messages judged relevant for this query.
    total = int(judgments.loc[judgments["query_id"].eq(query_id), "relevant"].sum())
    for method in ["lexical", "dense"]:
        found = with_judgments(rank_messages(query_id, method, k=10), query_id, "feedback_id")
        rows.append({
            "query_id": query_id,
            "method": method,
            "relevant_total": total,
            "P@5": precision_at_k(found["relevant"]),
            "RR": round(reciprocal_rank(found["relevant"]), 2),
        })
message_metrics = pd.DataFrame(rows)

# Mean P@5 and RR for each method over the four queries.
print(message_metrics.groupby("method")[["P@5", "RR"]].mean().round(2))
message_metrics

### 1.10 Read a failure

TQ04 asks "order has not arrived". Only FB014 is relevant. The lexical list below adds the words and phrases each message shares with the query. Which shared words pushed other messages ahead of FB014?

In [ ]:
# The analyzer splits a text into the same words and two-word phrases that TF-IDF uses.
analyze = msg_tfidf.build_analyzer()
query_terms = set(analyze(query_text["TQ04"]))

found = with_judgments(rank_messages("TQ04", "lexical", k=6), "TQ04", "feedback_id")

# For each message, the terms it shares with the query (& keeps what two sets have in common).
shared = []
for text in found["text"]:
    shared.append(sorted(query_terms & set(analyze(text))))
found["shared_terms"] = shared
found

### 1.11 Save your outputs

This writes two small CSV files to `outputs/m06`: the top 5 lists with judgments, and the metric table. The data files are not changed. After saving the notebook, review and commit the changes in GitHub Desktop.

In [ ]:
# Collect the top 5 lists of all queries and methods in one table.
ranking_rows = []
for query_id in ["TQ01", "TQ02", "TQ03", "TQ04"]:
    for method in ["lexical", "dense"]:
        found = with_judgments(rank_messages(query_id, method), query_id, "feedback_id")
        # insert(0, ...) adds a column at the front.
        found.insert(0, "method", method)
        found.insert(0, "query_id", query_id)
        ranking_rows.append(found)
pd.concat(ranking_rows).to_csv(OUTPUT_DIR / "feedback_rankings.csv", index=False)
message_metrics.to_csv(OUTPUT_DIR / "feedback_metrics.csv", index=False)

print("saved:", OUTPUT_DIR / "feedback_rankings.csv")
print("saved:", OUTPUT_DIR / "feedback_metrics.csv")

### Checkpoint

Answer in the cell below (2–3 sentences in total).

1. In 1.5, the lexical method ranked FB019 first for "money back for a broken product". Which word did it match, and why is FB019 not relevant?
2. FB005, "The refund arrived yesterday. No action is required.", is judged not relevant for TQ01. Do you agree? Use the information need in your answer.

**Your answer:**

## Part 2 Practice: searching real complaints

Now an analyst searches real complaints. The data is the course complaint file you met in Modules 2 and 3, `data/course/cfpb_complaints.csv`. It holds 10,000 complaints that consumers sent to the Consumer Financial Protection Bureau (CFPB). Each has a written `narrative`, a `product` and an `issue` chosen by the consumer, and a fixed `split`: 6,000 `train`, 2,000 `valid` and 2,000 `test`.

We search only the 6,000 `train` complaints, so the `valid` and `test` complaints stay unseen for later modules. Five complaint queries were set in advance: two worked examples (CQ01, CQ02) and three for your Practice (CQ03–CQ05).

### 2.1 Load the complaints and the chunk vectors

The embedding model reads at most 256 **tokens** (words or pieces of words). Many complaints are longer: 4,157 of the 10,000 would be cut. So I split each narrative into **chunks** of at most 150 words, saved in `data/processed/cfpb_chunks_150w.csv`. Then I embedded each chunk once with the same model and revision as the queries.

The chunk vectors are in `data/embeddings/cfpb_chunks_minilm.npy`, one per row of the chunk file, in the same order. Like all saved vectors in that folder, they come with an `_ids.csv` file and a `_manifest.json` file. A chunk ID is the complaint ID plus a chunk number, for example `14362649_2`.

In [ ]:
complaints = pd.read_csv(DATA / "course/cfpb_complaints.csv")
# One row per chunk, and one saved vector per chunk in the same order.
chunks = pd.read_csv(DATA / "processed/cfpb_chunks_150w.csv")
chunk_vectors = np.load(EMB / "cfpb_chunks_minilm.npy").astype("float32")
print("chunk vectors:", chunk_vectors.shape)

### 2.2 The chunking choice

This is a documented **configuration choice**: we search chunks, and a complaint's score is the score of its **best chunk**. A long complaint can then match on a detail near its end. A 150-word chunk can still exceed 256 tokens, because one word can become several tokens. Chunking reduces truncation; splitting by tokens would enforce the limit. The table counts training complaints by their number of chunks.

In [ ]:
# True for chunks of training complaints; use it to keep the same rows of the table and the vectors.
# .to_numpy() turns the pandas column into a plain numpy array, which can select rows of the vectors.
is_train = chunks["split"].eq("train").to_numpy()
train_chunks = chunks[is_train].reset_index(drop=True)
train_chunk_vectors = chunk_vectors[is_train]

# Add each complaint's product and issue to its chunks (how="left" keeps every chunk and its order).
complaint_labels = complaints[["complaint_id", "product", "issue"]]
train_chunks = train_chunks.merge(complaint_labels, on="complaint_id", how="left")
print("train complaints:", train_chunks["complaint_id"].nunique(), "| train chunks:", len(train_chunks))

# One row per complaint, then count complaints by their number of chunks.
one_row_per_complaint = train_chunks.drop_duplicates("complaint_id")
one_row_per_complaint["n_chunks"].value_counts().sort_index()

### 2.3 The lexical index

We fit TF-IDF on the training chunks. It keeps words and two-word phrases that appear in at least 5 chunks (`min_df=5`). The CFPB replaces private details with `XX` and `XXXX`, so these redaction marks are removed as stop words. Fitting takes a few seconds.

In [ ]:
chunk_tfidf = TfidfVectorizer(
    ngram_range=(1, 2),         # words and two-word phrases
    min_df=5,                   # keep terms that appear in at least 5 chunks
    stop_words=["xx", "xxxx"],  # drop the redaction marks
)
X_chunks = chunk_tfidf.fit_transform(train_chunks["text"])
X_chunks.shape

### 2.4 One search helper for both methods

`search` scores every training chunk, keeps the best chunk per complaint and returns the top `k` complaints. Set `method` to `"lexical"` or `"dense"`. The optional `product` argument is a **metadata filter**: it keeps only complaints with that product label. The `text` column shows the start of the best chunk. You will use this helper again in Module 12.

In [ ]:
def search(query_id, method="dense", k=10, product=None):
    """Take a query ID, "lexical" or "dense", and an optional product; return the top k complaints."""
    # Score every training chunk against the query, as in 1.5.
    if method == "lexical":
        query_tfidf = chunk_tfidf.transform([query_text[query_id]])
        scores = (X_chunks @ query_tfidf.T).toarray().ravel()
    else:
        scores = train_chunk_vectors @ query_vectors[query_row[query_id]]
    found = train_chunks[["complaint_id", "chunk_id", "product", "issue", "text"]].copy()
    found["score"] = scores

    # The metadata filter: keep only chunks of complaints with this product.
    if product is not None:
        found = found[found["product"].eq(product)]

    # Sort from the highest score down; kind="stable" keeps tied scores in file order.
    found = found.sort_values("score", ascending=False, kind="stable")
    # drop_duplicates keeps the first row per complaint, which is its best chunk.
    found = found.drop_duplicates("complaint_id").head(k)

    found.insert(0, "rank", range(1, len(found) + 1))
    found["score"] = found["score"].round(3)
    found["text"] = found["text"].str[:120]  # the first 120 characters of the best chunk
    return found.reset_index(drop=True)

def show_complaint(complaint_id):
    """Take a complaint ID; print its product, issue and full narrative."""
    # .iloc[0] takes the first (and only) matching row.
    row = complaints.loc[complaints["complaint_id"].eq(complaint_id)].iloc[0]
    print(f"{complaint_id} | {row['product']} | {row['issue']}")
    print(row["narrative"])

### 2.5 The complaint queries

Each query has an information need and a product for the filter configuration.

In [ ]:
# filter_product["CQ01"] gives the product used by the filter configuration.
filter_product = queries.set_index("query_id")["filter_product"]
queries.loc[queries["query_set"].eq("complaint"), ["query_id", "query", "information_need", "filter_product", "lab_use"]]

### 2.6 Worked example: two ranked lists for CQ02

CQ02: "scammer tricked me into sending money and the app will not refund it". Every complaint in these lists has a draft judgment. Read the passage and compare your reasoning with the `note` column. Compare the `relevant` column of the two lists.

In [ ]:
print("Information need:", need["CQ02"])
# The columns to show, then the lexical top 5 with judgments.
cols = ["rank", "complaint_id", "score", "product", "relevant", "text"]
with_judgments(search("CQ02", "lexical", k=5), "CQ02", "complaint_id")[cols]

In [ ]:
# The same query with dense search, and the same columns.
with_judgments(search("CQ02", "dense", k=5), "CQ02", "complaint_id")[cols]

### 2.7 Four configurations, one table

`evaluate` runs four configurations for one query: lexical or dense, each with and without the product filter. It reports P@5 and RR@10 (column `RR`). It also counts the results without a judgment yet: `not_judged_top5` for P@5 and `not_judged_top10` for RR@10, which uses all ten results. Read the metrics only when both counts are 0. For the worked examples they are 0.

In [ ]:
# Each configuration name maps to (method, use the product filter or not).
CONFIGS = {"lexical": ("lexical", False), "dense": ("dense", False),
           "lexical + filter": ("lexical", True), "dense + filter": ("dense", True)}

def evaluate(query_id):
    """Take a query ID; return one row per configuration with P@5, RR and the not-judged counts."""
    rows = []
    for name, (method, use_filter) in CONFIGS.items():
        if use_filter:
            product = filter_product[query_id]
        else:
            product = None
        found = with_judgments(search(query_id, method, k=10, product=product), query_id, "complaint_id")
        rows.append({
            "query_id": query_id,
            "config": name,
            "P@5": precision_at_k(found["relevant"]),
            "RR": round(reciprocal_rank(found["relevant"]), 2),
            # .isna() is True for results without a judgment; sum counts them.
            "not_judged_top5": int(found["relevant"].head(5).isna().sum()),
            # RR@10 uses all ten results, so count missing judgments there too.
            "not_judged_top10": int(found["relevant"].isna().sum()),
        })
    return pd.DataFrame(rows)

worked_metrics = pd.concat([evaluate("CQ01"), evaluate("CQ02")], ignore_index=True)
worked_metrics

### 2.8 What the product filter removes

A filter is a rule, not a judgment. Look at CQ01 with lexical search, without and with the filter for "Checking or savings account". The filter removes a credit card complaint that is not relevant. It also removes a relevant complaint about a money transfer company, because that company is not a bank.

In [ ]:
show_cols = ["rank", "complaint_id", "product", "relevant"]
without_filter = with_judgments(search("CQ01", "lexical", k=5), "CQ01", "complaint_id")
with_filter = with_judgments(search("CQ01", "lexical", k=5, product=filter_product["CQ01"]), "CQ01", "complaint_id")

print("Without the filter:")
print(without_filter[show_cols])
print("\nWith the filter:")  # \n starts a new line
print(with_filter[show_cols])

### 2.9 Save the worked-example outputs

In [ ]:
worked_metrics.to_csv(OUTPUT_DIR / "worked_example_metrics.csv", index=False)
print("saved:", OUTPUT_DIR / "worked_example_metrics.csv")

### 2.10 Review the judgments for your three queries

Most results for CQ03–CQ05 have a drafted judgment. Six pairs are left for you: their `relevant` value is empty and `judged_by` says `student`. The cell prints each information need and the full complaint. Read each one against the relevance rule in 1.7.

In [ ]:
# The six pairs left for you to judge.
to_judge = judgments[judgments["judged_by"].eq("student")]
# iterrows goes through a table one row at a time; _ is the row label, which we do not need.
for _, row in to_judge.iterrows():
    print(f"{row['query_id']} | need: {need[row['query_id']]}")
    show_complaint(int(row["doc_id"]))  # doc_id is stored as text, so turn it into a number
    print("-" * 80)  # a separator line

### Exercise 1: complete the judgments

Create a dictionary called `my_judgments`. Each key is a pair `(query_id, complaint_id)`, and each value is `1` (relevant) or `0` (not relevant). Judge all six pairs from 2.10. Example of the format (not an answer):

```python
my_judgments = {("CQ03", 11111111): 1, ("CQ03", 22222222): 0}
```

Then run the cell after yours. It adds your judgments and should report `pairs still to judge: 0`.

In [ ]:
# Exercise 1: your code here
# TODO: create a dictionary called my_judgments with one entry per pair:
#       key = (query_id, complaint_id), value = 1 (relevant) or 0 (not relevant).

In [ ]:
# This runs only after you have created my_judgments in Exercise 1.
if "my_judgments" in globals():
    for (query_id, complaint_id), value in my_judgments.items():
        # The row of the judgments table for this pair (doc_id is stored as text).
        pair_row = judgments["query_id"].eq(query_id) & judgments["doc_id"].eq(str(complaint_id))
        judgments.loc[pair_row, "relevant"] = value

# Count the pairs that still have no judgment.
print("pairs still to judge:", int(judgments["relevant"].isna().sum()))

### Exercise 2: metric table for CQ03, CQ04 and CQ05

Use `evaluate` for the three Practice queries and combine the results into one table called `practice_metrics`. Show the mean P@5 and RR per configuration. Save `practice_metrics` as `practice_metrics.csv` in `OUTPUT_DIR`.

Check `not_judged_top5` and `not_judged_top10`: both should be 0 everywhere once Exercise 1 is complete. Read P@5 and RR@10 only after that.

In [ ]:
# Exercise 2: your code here
# TODO: run evaluate(...) for each of the three queries and combine the three tables
#       with pd.concat(..., ignore_index=True). Call the result practice_metrics.
# TODO: print the mean P@5 and RR per configuration (group by "config").
# TODO: save practice_metrics as practice_metrics.csv in OUTPUT_DIR.

### Exercise 3: explain one failure

1. In your Exercise 2 table, choose one query and configuration with P@5 below 1.
2. Show its top 5 with `with_judgments(search(...), ...)` and print the full text of one result that is not relevant with `show_complaint`.
3. Test one targeted fix that this lab supports: turn the product filter on or off, or switch between lexical and dense. Report the P@5 before and after the fix for that query.

In [ ]:
# Exercise 3: your code here
# TODO: choose a query and configuration with P@5 below 1 in your Exercise 2 table.
# TODO: show its top 5 with with_judgments(search(...), query_id, "complaint_id").
# TODO: print the full text of one result that is not relevant with show_complaint(...).
# TODO: run the same query with one fix (filter on or off, or the other method),
#       and print P@5 before and after with precision_at_k(...).

### Practice interpretation (failure memo)

Write 3–5 sentences:

- Which configuration did best on your three queries, by P@5 and RR? Use your numbers.
- Describe the failure from Exercise 3: why did that complaint rank high, and what is the difference between it and a relevant result?
- Did your fix help, hurt or do nothing? Propose one targeted improvement for this query, and say what it could break.

**Your answer:**

### Submit

Run the whole notebook from the top (**Restart**, then **Run All**) and save it, so the outputs are saved in the file. Then commit and push it in GitHub Desktop before the deadline. You upload nothing to Canvas: the instructor grades the saved notebook in your repository as it is at the deadline.

If you used an AI tool to explain or debug code, add one line here: the tool, what you used it for, and how you checked the result.

**AI use:**

## Part 3 Challenge (optional, not graded)

Run these parts after the required work, in order. Part A is the reranking from the video. You can also use `reranked(query_id)` from A1 as one more fix to test for your Exercise 3 query.

### A. Cached cross-encoder reranking

#### A1. Load the reranker scores

A **cross-encoder** reads the query and one text **together** and returns a relevance score. That is more careful than comparing two separate vectors, but it must run once for every pair, so it is too slow to score every chunk. The usual design is **retrieve, then rerank**: take a small candidate pool from fast search, then rerank only that pool.

For each complaint query, I built the pool from the lexical top 20 and the dense top 20 complaints. Then I scored every chunk in the pool with `cross-encoder/ms-marco-MiniLM-L6-v2`. A complaint's reranker score is its best chunk's score. Higher means more relevant; the scores are not probabilities. In the table, `lexical_rank` or `dense_rank` is empty when the complaint was outside that top 20.

In [ ]:
# One row per (query, chunk) pair that the reranker scored.
rerank = pd.read_csv(RET / "m06_rerank_scores.csv")
print("model:", rerank["model"].unique()[0], "| revision:", rerank["model_revision"].unique()[0][:7])

# Number of scored pairs for each complaint query.
complaint_pairs = rerank[rerank["query_set"].eq("complaint")]
print("pairs per complaint query:", complaint_pairs.groupby("query_id").size().to_dict())

def reranked(query_id, k=10):
    """Take a query ID; return the top k complaints of its pool by their best reranker score."""
    pool = rerank[rerank["query_id"].eq(query_id)].copy()
    pool["complaint_id"] = pool["complaint_id"].astype("int64")
    # One row per complaint: its best (max) chunk score and its ranks in the two first lists.
    best = pool.groupby("complaint_id").agg(
        rerank_score=("rerank_score", "max"),
        lexical_rank=("lexical_rank", "first"),
        dense_rank=("dense_rank", "first"),
    )
    best = best.sort_values("rerank_score", ascending=False).head(k).reset_index()
    best.insert(0, "rank", range(1, len(best) + 1))
    # Whole-number ranks that can be empty (outside that top 20).
    best[["lexical_rank", "dense_rank"]] = best[["lexical_rank", "dense_rank"]].astype("Int64")
    return best

with_judgments(reranked("CQ02"), "CQ02", "complaint_id")

#### A2. Before and after reranking

Compare dense search (before) with the reranked lexical + dense pool (after) on CQ01 and CQ02. The cell also saves the table as `rerank_metrics.csv`. Part of any gain can come from the wider pool: it adds lexical results that dense search ranked low or missed.

In [ ]:
rows = []
for query_id in ["CQ01", "CQ02"]:
    # The dense top 10 (before) and the reranked pool (after).
    before = with_judgments(search(query_id, "dense", k=10), query_id, "complaint_id")
    after = with_judgments(reranked(query_id), query_id, "complaint_id")
    for name, found in [("dense", before), ("reranked pool", after)]:
        rows.append({
            "query_id": query_id,
            "list": name,
            "P@5": precision_at_k(found["relevant"]),
            "RR": round(reciprocal_rank(found["relevant"]), 2),
        })
rerank_metrics = pd.DataFrame(rows)
rerank_metrics.to_csv(OUTPUT_DIR / "rerank_metrics.csv", index=False)
rerank_metrics

#### A3. A reranking failure

For CQ02 the reranker put complaint 14904289 first. Read it. The consumer sent money to the wrong person by mistake: the words "refund", "sent money" and "Cash App" fit the query, but there was no scam. The reranker reads the text more closely than a vector does, and it can still be wrong.

In [ ]:
show_complaint(14904289)

# The note that explains the judgment for this pair.
pair_row = judgments["query_id"].eq("CQ02") & judgments["doc_id"].eq("14904289")
print("\nJudgment note:", judgments.loc[pair_row, "note"].iloc[0])

#### A4. What reranking costs

When I ran the reranker, it took about 11.6 milliseconds per pair on a CPU. The cell times your dense search and estimates the reranker's time. Your search time depends on your computer.

In [ ]:
RERANK_MS_PER_PAIR = 11.6  # measured when the reranker scores were saved

# Time 10 dense searches and take the average, in milliseconds.
start = time.perf_counter()
for _ in range(10):
    search("CQ02", "dense")
dense_ms = (time.perf_counter() - start) / 10 * 1000

# Estimated reranker time in seconds: number of pairs times the time per pair.
pool_pairs = int(rerank["query_id"].eq("CQ02").sum())
pool_seconds = pool_pairs * RERANK_MS_PER_PAIR / 1000
all_seconds = len(train_chunks) * RERANK_MS_PER_PAIR / 1000

# In the f-strings, :, adds thousands separators, :.0f shows no decimals and :.1f one decimal.
print(f"dense search over {len(train_chunks):,} chunks: about {dense_ms:.0f} ms per query")
print(f"reranking the CQ02 pool ({pool_pairs} pairs): about {pool_seconds:.1f} s")
print(f"reranking every training chunk: about {all_seconds:.0f} s per query")

### B. Hybrid retrieval with reciprocal rank fusion

**Hybrid** retrieval combines a lexical and a dense list. Reciprocal rank fusion (RRF) gives each complaint `1 / (60 + rank)` from each list and adds the parts. It uses ranks, not scores, so the two score scales do not need to match. Results without a judgment are counted in `not_judged_top5`.

In [ ]:
def hybrid(query_id, k=10, c=60):
    """Take a query ID; return the top k complaints by reciprocal rank fusion of the two top 20 lists."""
    fused = {}  # complaint ID -> RRF score
    for method in ["lexical", "dense"]:
        found = search(query_id, method, k=20)
        # zip walks through the two columns side by side.
        for rank, complaint_id in zip(found["rank"], found["complaint_id"]):
            if complaint_id not in fused:
                fused[complaint_id] = 0
            # A complaint in both lists gets one part from each list.
            fused[complaint_id] = fused[complaint_id] + 1 / (c + rank)
    # Put the fused scores in a table and sort it, highest first.
    # kind="stable" keeps tied scores in the order the complaints were first seen.
    table = pd.DataFrame({"complaint_id": list(fused.keys()), "rrf_score": list(fused.values())})
    table = table.sort_values("rrf_score", ascending=False, kind="stable").head(k)
    table["rrf_score"] = table["rrf_score"].round(4)
    table.insert(0, "rank", range(1, len(table) + 1))
    return table.reset_index(drop=True)

rows = []
for query_id in ["CQ01", "CQ02", "CQ03", "CQ04", "CQ05"]:
    found = with_judgments(hybrid(query_id), query_id, "complaint_id")
    rows.append({
        "query_id": query_id,
        "P@5 hybrid": precision_at_k(found["relevant"]),
        "RR hybrid": round(reciprocal_rank(found["relevant"]), 2),
        "not_judged_top5": int(found["relevant"].head(5).isna().sum()),
    })
pd.DataFrame(rows)

### C. Whole complaints or chunks?

The file `data/embeddings/cfpb_minilm.npy` holds one vector per complaint, in the row order of `cfpb_complaints.csv`, made with the same model and revision. Each vector sees only the first 256 tokens of its complaint. Repeat dense search with these whole-complaint vectors and compare P@5 with the chunk search.

The whole-complaint search can return complaints that nobody has judged yet. `precision_at_k` counts those as not relevant, so a lower P@5 may only mean missing judgments. The table shows `not_judged` for each top 5. Before you compare P@5, judge every missing pair with the rule in 1.7.

In [ ]:
# One vector per complaint; keep the training complaints, numbered 0, 1, 2, ...
cfpb_vectors = np.load(EMB / "cfpb_minilm.npy").astype("float32")
whole_train = complaints["split"].eq("train").to_numpy()
train_complaints = complaints[whole_train].reset_index(drop=True)
train_complaint_vectors = cfpb_vectors[whole_train]

rows = []
for query_id in ["CQ01", "CQ02", "CQ03", "CQ04", "CQ05"]:
    scores = train_complaint_vectors @ query_vectors[query_row[query_id]]
    # np.argsort(-scores)[:10] gives the positions of the 10 highest scores.
    top_rows = np.argsort(-scores)[:10]
    top = train_complaints.loc[top_rows, ["complaint_id"]].reset_index(drop=True)
    whole = with_judgments(top, query_id, "complaint_id")
    chunk = with_judgments(search(query_id, "dense"), query_id, "complaint_id")
    rows.append({
        "query_id": query_id,
        "P@5 whole": precision_at_k(whole["relevant"]),
        "not_judged whole": int(whole["relevant"].head(5).isna().sum()),
        "P@5 chunks": precision_at_k(chunk["relevant"]),
        "not_judged chunks": int(chunk["relevant"].head(5).isna().sum()),
    })
pd.DataFrame(rows)

### D. Run a cross-encoder yourself

This part needs the `sentence-transformers` package, which is not in the course environment, and downloads the reranker model (about 90 MB) the first time. Install it with `pip install sentence-transformers` in the `ban6403` environment if you want to try it. Then set `RUN_CROSS_ENCODER = True` in the cell below. The cell loads the same model revision that made the saved scores, so your scores should be close to the saved ones.

The switch stays `False` by default, so Run All never downloads a model.

In [ ]:
# Change to True only when you want to run Challenge D (it downloads the model).
RUN_CROSS_ENCODER = False

if not RUN_CROSS_ENCODER:
    print("Challenge D is switched off. This part is optional.")
else:
    # The package is optional, so the cell prints a note instead of failing when it is missing.
    try:
        from sentence_transformers import CrossEncoder
    except ImportError:
        print("sentence-transformers is not installed. This part is optional.")
    else:
        # The saved scores record the model name and revision; load exactly that version.
        model = CrossEncoder(rerank["model"].iloc[0], revision=rerank["model_revision"].iloc[0])
        sample = rerank[rerank["query_id"].eq("CQ02")].head(3)
        # The model reads (query, text) pairs and returns one score per pair.
        mine = model.predict(list(zip(sample["query"], sample["text"])))
        print(pd.DataFrame({"doc_id": sample["doc_id"], "saved": sample["rerank_score"], "mine": np.round(mine, 4)}))

## Further reading

- [Introduction to Information Retrieval, chapter 8: Evaluation in information retrieval](https://nlp.stanford.edu/IR-book/html/htmledition/evaluation-in-information-retrieval-1.html)
- [Sentence Transformers: retrieve and re-rank](https://sbert.net/examples/sentence_transformer/applications/retrieve_rerank/README.html)
- [Model card: cross-encoder/ms-marco-MiniLM-L6-v2](https://huggingface.co/cross-encoder/ms-marco-MiniLM-L6-v2)

Next, Module 7 analyzes customer feedback and asks whether one sentiment score can show which product feature needs action.